# 04 · Evaluación con la regla pre-registrada

**Etapa 7 del plan.** Carga **por primera vez** la clase de 201909 y 201910, calcula la ganancia de las 100 predicciones del `03` y **aplica la regla de la sección 6 del pre-registro** (`docs/diseno_experimental.md`), con las precisiones de la sección 11. También arma las 4 figuras del video (sección 9).

**Se corre una sola vez.** No entrena ni elige nada: N sale de `resultados/03_N.csv`, que se eligió en 201908. Si aparece un error de código (no un resultado que no guste), se corrige, se registra en la sección 11 y se vuelve a correr.

**Antes de correrlo:**
- el `03` tiene que estar verificado;
- este notebook, los `resultados/03_*` y la sección 11 del pre-registro tienen que estar commiteados.

Ficha: `fichas/etapa_7_evaluacion.md` (en el laboratorio).

## 1. Parámetros

Copiados del pre-registro. **No se cambian después de ver los resultados.**

In [ ]:
from pathlib import Path
import warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
EXP_03 = Path('../exp/03')                          # solo se lee
RESULTADOS = Path('../resultados')
FIGURAS = RESULTADOS / 'figuras'

CONTROL = 'A'
VARIANTES_REGLA = ['L3', 'L4', 'T3', 'T4', 'LT3', 'LT4']
FAMILIAS = {'L': ['A', 'L3', 'L4'], 'T': ['A', 'T3', 'T4'], 'LT': ['A', 'LT3', 'LT4']}   # h = 2, 3, 4
RANKINGS = ('LT4-ranking', 'A-ranking')              # secundario: variante, control
REFERENCIA = 'A-consigna'                            # solo descriptiva
CONFIGURACIONES = [CONTROL] + VARIANTES_REGLA + list(RANKINGS[::-1]) + [REFERENCIA]

MES_VALID = 201908
MES_PRINCIPAL = 201910
MES_SECUNDARIO = 201909
MESES_EVAL = [MES_VALID, MES_SECUNDARIO, MES_PRINCIPAL]

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 10
ALFA = 0.05
BANDA = (3_000, 6_000)                               # ambos incluidos
NIVEL_IC = 0.95

GANANCIA_ACIERTO = 1_072_500                         # provisorio
COSTO_ESTIMULO = 27_500
VENTANA_SUAVIZADO = 501
LARGO_CURVA = 25_000
# -----------------------------------------------------------------------------

PRED = EXP_03 / 'pred'
CLAVES_EVAL = EXP_03 / 'claves_eval.npz'
VALIDACION_CSV = RESULTADOS / '03_validacion.csv'
N_CSV = RESULTADOS / '03_N.csv'
GANANCIAS_CSV = RESULTADOS / '04_ganancias.csv'
REGLA_CSV = RESULTADOS / '04_regla.csv'
DOSIS_CSV = RESULTADOS / '04_dosis.csv'
SECUNDARIOS_CSV = RESULTADOS / '04_secundarios.csv'
FIGURAS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14

ESPERADO = {201908: (131_775, 543), 201909: (133_419, 615), 201910: (135_633, 632)}   # filas, BAJA+2


def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


SEMILLAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()[:N_SEMILLAS]
assert SEMILLAS == [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]
print('Configuraciones:', CONFIGURACIONES)

## 2. Funciones de la regla, con sus controles

- **Wilcoxon exacto** (precisión b): se enumeran las 2ⁿ asignaciones de signo (1.024 con 10 pares). Si hay empates en |d|, usa rangos promedio; las diferencias iguales a 0 se descartan. Hace falta porque, con N fijo, la ganancia se mueve de a saltos de 1,1 M y las diferencias pueden empatar.
- **Holm** sobre los 6 p-valores.
- **IC 95 %** de la media entre semillas, con t de Student (precisión a).

Los `assert` comprueban cada función con casos de resultado conocido.

In [ ]:
def wilcoxon_exacto(d):
    """p-valor de dos colas del test de rangos con signo, enumerando todos los signos."""
    #d = np.asarray(d, dtype=np.float64)
    d = np.round(np.asarray(d, dtype=np.float64), 6)
    d = d[d != 0]
    n = len(d)
    if n == 0:
        return 1.0
    rangos = stats.rankdata(np.abs(d))
    w = float((np.sign(d) * rangos).sum())
    signos = 1 - 2 * ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1)
    return float(np.mean(np.abs(signos @ rangos) >= abs(w) - 1e-9))


def holm(p):
    p = np.asarray(p, dtype=np.float64)
    m = len(p)
    orden = np.argsort(p, kind='stable')
    ajustado = np.minimum(1.0, np.maximum.accumulate((m - np.arange(m)) * p[orden]))
    salida = np.empty(m)
    salida[orden] = ajustado
    return salida


def ic(x, nivel=NIVEL_IC):
    x = np.asarray(x, dtype=np.float64)
    media = x.mean()
    medio = stats.t.ppf((1 + nivel) / 2, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return media, media - medio, media + medio


# Controles (verificación 5 y 6 de la ficha)
np.testing.assert_allclose(holm([0.01, 0.04, 0.03, 0.005]), [0.03, 0.06, 0.06, 0.02])
assert abs(wilcoxon_exacto(np.arange(1, 11)) - 2 / 1024) < 1e-12           # 10 a favor: p = 0,00195
assert abs(wilcoxon_exacto(-np.arange(1, 11)) - 2 / 1024) < 1e-12
rng = np.random.default_rng(SEMILLA_PRIMIGENIA)
for _ in range(20):                                                        # sin empates: igual que scipy
    d = rng.normal(size=10)
    assert abs(wilcoxon_exacto(d) - stats.wilcoxon(d, method='exact').pvalue) < 1e-12
assert abs(wilcoxon_exacto([1, 1, 1, 1, 1, 1, 1, 1, 1, -1]) - 22 / 1024) < 1e-12   # empates: a mano

assert abs(wilcoxon_exacto([0.1 + 0.2, -0.3, 1, 2, 3, 4, 5, 6, 7, 8])
           - wilcoxon_exacto([0.3, -0.3, 1, 2, 3, 4, 5, 6, 7, 8])) < 1e-12   # empates con ruido de float

media, inf, sup = ic(np.arange(10))
assert abs(media - 4.5) < 1e-12 and abs(sup - 4.5 - 2.262157 * np.std(np.arange(10), ddof=1) / np.sqrt(10)) < 1e-5
print('Holm, Wilcoxon e IC: controles OK')

## 3. Carga: predicciones del `03` y, por primera vez, la clase de 201909 y 201910

La clase se une por (cliente, mes) en el orden de las claves que guardó el `03`.

In [ ]:
tabla_n = pd.read_csv(N_CSV).set_index('config')
validacion = pd.read_csv(VALIDACION_CSV)
assert set(CONFIGURACIONES) <= set(tabla_n.index), set(CONFIGURACIONES) - set(tabla_n.index)
assert (tabla_n.loc[CONFIGURACIONES, 'semillas'] == N_SEMILLAS).all()
N = tabla_n['N'].astype(int).to_dict()

claves = np.load(CLAVES_EVAL)
claves = pd.DataFrame({'numero_de_cliente': claves['numero_de_cliente'], 'foto_mes': claves['foto_mes']})
PROB = {c: np.stack([np.load(PRED / c / f'{s}.npy') for s in SEMILLAS]) for c in CONFIGURACIONES}
assert all(p.shape == (N_SEMILLAS, len(claves)) for p in PROB.values())
assert all(np.isfinite(p).all() for p in PROB.values())

con = duckdb.connect()
clase = con.execute(f"""
    select numero_de_cliente, foto_mes, clase_ternaria
    from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes in ({', '.join(str(m) for m in MESES_EVAL)})
""").df()
con.close()
eval_ = claves.merge(clase, on=['numero_de_cliente', 'foto_mes'], how='left', validate='one_to_one')
assert len(eval_) == len(claves) and eval_['clase_ternaria'].notna().all()
MES = eval_['foto_mes'].to_numpy()
ES_B2 = (eval_['clase_ternaria'] == 'BAJA+2').to_numpy()

for mes, (filas, b2) in ESPERADO.items():                 # verificación 1
    assert ((MES == mes).sum(), ES_B2[MES == mes].sum()) == (filas, b2), mes
print(f'{len(PROB)} configuraciones × {N_SEMILLAS} semillas = {len(PROB) * N_SEMILLAS} predicciones')   # verificación 2
print(pd.DataFrame({m: {'filas': f, 'BAJA+2': b} for m, (f, b) in ESPERADO.items()}).to_string())

## 4. Control de alineación (201908)

Si las predicciones y la clase quedaron bien unidas, recalcular 201908 desde cero tiene que dar **lo mismo que el `03`**: la AUC de cada semilla, el N de cada configuración y la ganancia en ese N.

In [ ]:
def curva(prob, es_b2):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:LARGO_CURVA].astype(np.float64)


es_mes = {m: MES == m for m in MESES_EVAL}
CURVAS = {(c, m): np.stack([curva(PROB[c][i, es_mes[m]], ES_B2[es_mes[m]]) for i in range(N_SEMILLAS)])
          for c in CONFIGURACIONES for m in MESES_EVAL}
AUC = {(c, m): [roc_auc_score(ES_B2[es_mes[m]], PROB[c][i, es_mes[m]]) for i in range(N_SEMILLAS)]
       for c in CONFIGURACIONES for m in MESES_EVAL}

val = validacion.drop_duplicates(['config', 'semilla'], keep='last').set_index(['config', 'semilla'])
for c in CONFIGURACIONES:                                 # verificación 3
    suave = pd.Series(CURVAS[(c, MES_VALID)].mean(axis=0)).rolling(
        VENTANA_SUAVIZADO, center=True, min_periods=VENTANA_SUAVIZADO).mean().to_numpy()
    assert int(np.nanargmax(suave)) + 1 == N[c], (c, int(np.nanargmax(suave)) + 1, N[c])
    for i, s in enumerate(SEMILLAS):
        assert abs(AUC[(c, MES_VALID)][i] - val.loc[(c, s), 'auc_201908']) < 6e-7, (c, s)
        g = CURVAS[(c, MES_VALID)][i, N[c] - 1] / 1e6
        assert abs(g - val.loc[(c, s), 'ganancia_201908_en_N_M']) < 6e-4, (c, s)
print('Alineación OK: la AUC, el N y la ganancia de 201908 coinciden con el 03 en las', len(CONFIGURACIONES), 'configuraciones')

## 5. Ganancia y AUC por configuración, semilla y mes

- **Ganancia:** el valor de la curva sin suavizar en el N de la configuración (los envíos reales).
- **Banda:** el promedio de la curva entre N = 3.000 y N = 6.000, ambos incluidos (precisión c).
- **AUC:** contra `BAJA+2`; solo se reporta (precisión d).

In [ ]:
filas = []
for c in CONFIGURACIONES:
    for m in MESES_EVAL:
        for i, s in enumerate(SEMILLAS):
            cv = CURVAS[(c, m)][i]
            filas.append({'config': c, 'semilla': s, 'foto_mes': m, 'N': N[c],
                          'ganancia_M': round(cv[N[c] - 1] / 1e6, 4),
                          'ganancia_banda_M': round(cv[BANDA[0] - 1:BANDA[1]].mean() / 1e6, 4),
                          'auc': round(AUC[(c, m)][i], 6)})
ganancias = pd.DataFrame(filas)
ganancias.to_csv(GANANCIAS_CSV, index=False)

for m in (MES_SECUNDARIO, MES_PRINCIPAL):                # verificación 4
    tope = ESPERADO[m][1] * GANANCIA_ACIERTO / 1e6
    assert ganancias.loc[ganancias['foto_mes'] == m, 'ganancia_M'].max() < tope, m
    print(f'{m}: ganancia máxima posible {tope:.1f} M · tope respetado')

G = ganancias.pivot_table(index=['config', 'foto_mes'], columns='semilla', values='ganancia_M')[SEMILLAS]
B = ganancias.pivot_table(index=['config', 'foto_mes'], columns='semilla', values='ganancia_banda_M')[SEMILLAS]
resumen = ganancias.groupby(['config', 'foto_mes']).agg(ganancia_media_M=('ganancia_M', 'mean'),
                                                       auc_media=('auc', 'mean')).unstack('foto_mes')
print(resumen.loc[CONFIGURACIONES].round(4).to_string())

## 6. La regla (sección 6 del pre-registro)

Para cada variante V: d = ganancia de V − ganancia de A, por semilla, en 201910. Wilcoxon de dos colas, Holm sobre las 6 y α = 0,05.

**V le gana a A si cumple las tres:** p de Holm < 0,05; diferencia media en 201910 > 0; diferencia media en 201909 ≥ 0.

**Robustez:** si la diferencia media con la banda tiene el signo contrario al de la diferencia media principal, el resultado es **frágil**.

In [ ]:
def comparar(v, control):
    #d10 = (G.loc[(v, MES_PRINCIPAL)] - G.loc[(control, MES_PRINCIPAL)]).to_numpy()
    d10 = (G.loc[(v, MES_PRINCIPAL)] - G.loc[(control, MES_PRINCIPAL)]).to_numpy().round(6)

    d09 = (G.loc[(v, MES_SECUNDARIO)] - G.loc[(control, MES_SECUNDARIO)]).to_numpy()
    db = (B.loc[(v, MES_PRINCIPAL)] - B.loc[(control, MES_PRINCIPAL)]).to_numpy()
    media, inf, sup = ic(d10)
    return {'variante': v, 'control': control, 'N_variante': N[v], 'N_control': N[control],
            'd_media_201910_M': media, 'ic95_inf_M': inf, 'ic95_sup_M': sup,
            'semillas_a_favor': int((d10 > 0).sum()), 'p': wilcoxon_exacto(d10),
            'd_media_201909_M': d09.mean(), 'd_banda_201910_M': db.mean(),
            'robustez': 'frágil' if np.sign(db.mean()) != np.sign(media) else 'robusto'}


regla = pd.DataFrame([comparar(v, CONTROL) for v in VARIANTES_REGLA])
regla['p_holm'] = holm(regla['p'])
regla['gana'] = (regla['p_holm'] < ALFA) & (regla['d_media_201910_M'] > 0) & (regla['d_media_201909_M'] >= 0)
regla = regla[['variante', 'control', 'N_variante', 'N_control', 'd_media_201910_M', 'ic95_inf_M', 'ic95_sup_M',
               'semillas_a_favor', 'p', 'p_holm', 'd_media_201909_M', 'd_banda_201910_M', 'robustez', 'gana']]
regla.round(4).to_csv(REGLA_CSV, index=False)
print(regla.round(4).to_string(index=False))

## 7. Dosis-respuesta y análisis secundarios

- **Dosis-respuesta:** ganancia media en 201910 (y 201909) según h = 2 (A), 3 y 4, en las familias L, T y LT, con IC 95 %. Se reporta siempre.
- **Rankings:** LT4-ranking contra A-ranking, con el mismo test, **sin corrección** y fuera de la regla.
- **A-consigna:** solo descriptiva, sin test.

In [ ]:
filas = []
for familia, configs in FAMILIAS.items():
    for h, c in zip((2, 3, 4), configs):
        for m in (MES_SECUNDARIO, MES_PRINCIPAL):
            media, inf, sup = ic(G.loc[(c, m)])
            filas.append({'familia': familia, 'h': h, 'config': c, 'foto_mes': m,
                          'ganancia_media_M': media, 'ic95_inf_M': inf, 'ic95_sup_M': sup})
dosis = pd.DataFrame(filas)
dosis.round(4).to_csv(DOSIS_CSV, index=False)
print(dosis[dosis['foto_mes'] == MES_PRINCIPAL].round(3).to_string(index=False))

rankings = comparar(*RANKINGS)
filas = [{'analisis': 'rankings (sin corrección)', 'config': RANKINGS[0], 'contra': RANKINGS[1], 'medida': k, 'valor': v}
         for k, v in rankings.items() if k not in ('variante', 'control')]
for m in (MES_SECUNDARIO, MES_PRINCIPAL):
    for c in (REFERENCIA, CONTROL):
        media, inf, sup = ic(G.loc[(c, m)])
        filas += [{'analisis': 'A-consigna (descriptiva)', 'config': c, 'contra': '', 'medida': f'{k}_{m}_M', 'valor': x}
                  for k, x in (('ganancia_media', media), ('ic95_inf', inf), ('ic95_sup', sup))]
    filas.append({'analisis': 'A-consigna (descriptiva)', 'config': REFERENCIA, 'contra': CONTROL,
                  'medida': f'd_media_{m}_M', 'valor': (G.loc[(REFERENCIA, m)] - G.loc[(CONTROL, m)]).mean()})
secundarios = pd.DataFrame(filas)
secundarios.assign(valor=lambda t: t['valor'].map(lambda x: round(x, 4) if isinstance(x, float) else x)).to_csv(
    SECUNDARIOS_CSV, index=False)
print()
print(secundarios.to_string(index=False))

## 8. Figuras del video (sección 9 del pre-registro)

1. Ganancia por semilla en 201910: A contra cada variante, con el resultado de la regla.
2. Dosis-respuesta: ganancia media según h, una línea por familia, con IC 95 %.
3. Curvas de ganancia promedio en 201910 según N, para A y la mejor variante, con el N elegido en 201908. **La mejor variante** es la de mayor diferencia media en 201910 (precisión e; es solo para la figura).
4. La tabla de la regla aplicada.

Tamaño 1920 × 1080 (16:9), para el video.

In [ ]:
TINTA, TINTA_2, GRIS, GRILLA = '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
AZUL, NARANJA, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': TINTA_2, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2, 'text.color': TINTA,
                     'axes.grid': True, 'grid.color': GRILLA, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 13, 'axes.titlesize': 15})
TAMANO, DPI = (12.8, 7.2), 150


def coma(x, decimales=1):
    return f'{x:,.{decimales}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


millones = FuncFormatter(lambda x, _: coma(x, 0))
p_texto = lambda p: coma(p, 4) if p < 0.01 else coma(p, 3)
fila_regla = regla.set_index('variante')

# Figura 1: ganancia por semilla, A contra cada variante
fig, ejes = plt.subplots(2, 3, figsize=TAMANO, sharey=True)
for ax, v in zip(ejes.flat, VARIANTES_REGLA):
    ga, gv = G.loc[(CONTROL, MES_PRINCIPAL)].to_numpy(), G.loc[(v, MES_PRINCIPAL)].to_numpy()
    for a, b in zip(ga, gv):
        color = AZUL if b > a else NARANJA
        ax.plot([0, 1], [a, b], color=color, lw=1.5, alpha=0.8, marker='o', ms=6, mec='#fcfcfb', mew=1.5)
    ax.plot([0, 1], [ga.mean(), gv.mean()], color=TINTA, lw=2.5, marker='D', ms=8, zorder=5)
    r = fila_regla.loc[v]
    ax.set_title(f'{v}: {"GANA" if r["gana"] else "no gana"} · p Holm {p_texto(r["p_holm"])}', loc='left')
    ax.set_xticks([0, 1], [CONTROL, v])
    ax.set_xlim(-0.25, 1.25)
    ax.yaxis.set_major_formatter(millones)
for ax in ejes[:, 0]:
    ax.set_ylabel('Ganancia en 201910 (M$)')
fig.legend(handles=[plt.Line2D([], [], color=AZUL, marker='o', label='semilla a favor de la variante'),
                    plt.Line2D([], [], color=NARANJA, marker='o', label='semilla a favor de A'),
                    plt.Line2D([], [], color=TINTA, marker='D', lw=2.5, label='media de las 10 semillas')],
           loc='lower center', ncol=3, frameon=False)
fig.suptitle('Ganancia por semilla en 201910: A contra cada variante', x=0.02, ha='left', fontsize=17)
fig.tight_layout(rect=(0, 0.06, 1, 0.95))
fig.savefig(FIGURAS / '04_1_ganancia_por_semilla.png', dpi=DPI)
plt.show()

# Figura 2: dosis-respuesta
fig, ax = plt.subplots(figsize=TAMANO)
for (familia, color), dx in zip({'L': AZUL, 'T': NARANJA, 'LT': AQUA}.items(), (-0.06, 0, 0.06)):
    t = dosis[(dosis['familia'] == familia) & (dosis['foto_mes'] == MES_PRINCIPAL)]
    x = t['h'] + dx
    ax.errorbar(x, t['ganancia_media_M'], yerr=[t['ganancia_media_M'] - t['ic95_inf_M'], t['ic95_sup_M'] - t['ganancia_media_M']],
                color=color, lw=2, marker='o', ms=9, capsize=5, label=familia)
    ax.annotate(familia, (x.iloc[-1], t['ganancia_media_M'].iloc[-1]), xytext=(12, 0), textcoords='offset points',
                va='center', color=TINTA_2, fontsize=14)
ax.set_xticks([2, 3, 4], ['h = 2 (A)', 'h = 3', 'h = 4'])
ax.set_xlim(1.7, 4.4)
ax.yaxis.set_major_formatter(millones)
ax.set_ylabel('Ganancia media en 201910 (M$), IC 95 %')
ax.set_title('Dosis-respuesta: ganancia según los meses de historia', loc='left')
ax.legend(title='Familia', frameon=False, loc='upper left')
fig.tight_layout()
fig.savefig(FIGURAS / '04_2_dosis_respuesta.png', dpi=DPI)
plt.show()

# Figura 3: curvas promedio en 201910 para A y la mejor variante
mejor = regla.sort_values('d_media_201910_M', ascending=False)['variante'].iloc[0]
fig, ax = plt.subplots(figsize=TAMANO)
x = np.arange(1, LARGO_CURVA + 1)
par = (CONTROL, mejor)
for c, color in zip(par, (GRIS, AZUL)):
    media = CURVAS[(c, MES_PRINCIPAL)].mean(axis=0) / 1e6
    ax.plot(x, media, color=color, lw=2, label=f'{c} (media de 10 semillas)')
    ax.axvline(N[c], color=color, lw=1.2, ls='--')
    a_la_derecha = N[c] == max(N[p] for p in par)
    ax.text(N[c] + (120 if a_la_derecha else -120), 0.97, f'N {c} = {coma(N[c], 0)}', transform=ax.get_xaxis_transform(),
            ha='left' if a_la_derecha else 'right', va='top', color=TINTA_2, fontsize=12)
ax.axvspan(*BANDA, color=GRILLA, alpha=0.5, lw=0, label='banda de robustez (3.000 a 6.000)')
ax.set_xlim(0, 15_000)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: coma(v, 0)))
ax.yaxis.set_major_formatter(millones)
ax.set_xlabel('Cantidad de envíos (N)')
ax.set_ylabel('Ganancia en 201910 (M$)')
ax.set_title(f'Curva de ganancia en 201910: A y {mejor} (N elegido en 201908)', loc='left')
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(FIGURAS / '04_3_curvas_201910.png', dpi=DPI)
plt.show()

# Figura 4: tabla de la regla aplicada
celdas = [[r.variante, coma(r.d_media_201910_M), f'{coma(r.ic95_inf_M)} a {coma(r.ic95_sup_M)}', f'{r.semillas_a_favor}/10',
           p_texto(r.p), p_texto(r.p_holm), coma(r.d_media_201909_M), coma(r.d_banda_201910_M), r.robustez,
           'sí' if r.gana else 'no'] for r in regla.itertuples()]
columnas = ['Variante', 'Dif. 201910\n(M$)', 'IC 95 %\n(M$)', 'Semillas\na favor', 'p', 'p Holm', 'Dif. 201909\n(M$)',
            'Dif. banda\n(M$)', 'Robustez', 'Gana']
anchos = [0.08, 0.11, 0.15, 0.09, 0.08, 0.08, 0.11, 0.11, 0.1, 0.06]
fig, ax = plt.subplots(figsize=TAMANO)
ax.axis('off')
tabla = ax.table(cellText=celdas, colLabels=columnas, colWidths=anchos, cellLoc='center', bbox=(0, 0.12, 1, 0.74))
tabla.auto_set_font_size(False)
tabla.set_fontsize(13)
for (i, j), celda in tabla.get_celld().items():
    celda.set_edgecolor(GRILLA)
    if i == 0:
        celda.set_text_props(color=TINTA_2, weight='bold', fontsize=12)
ax.set_title('Regla pre-registrada: cada variante contra A (Wilcoxon + Holm, α = 0,05)', loc='left', fontsize=17)
fig.text(0.02, 0.04, 'Gana si p Holm < 0,05, la diferencia media en 201910 es > 0 y la de 201909 es ≥ 0. '
         'Frágil: la banda da el signo contrario.', color=TINTA_2, fontsize=12)
fig.savefig(FIGURAS / '04_4_tabla_regla.png', dpi=DPI)
plt.show()
print('Figuras en', FIGURAS)

## 9. Resultado en palabras

In [ ]:
ganadoras = regla.loc[regla['gana'], 'variante'].tolist()
if ganadoras:
    for r in regla[regla['gana']].itertuples():
        print(f'{r.variante} le gana a A: +{coma(r.d_media_201910_M, 2)} M de media en 201910 '
              f'({r.semillas_a_favor} de 10 semillas a favor, p Holm {p_texto(r.p_holm)}), '
              f'{coma(r.d_media_201909_M, 2)} M en 201909 · {r.robustez}.')
else:
    #print('Ninguna variante cumple la regla: más de 2 meses de historia no suma en esta ventana.')
    print('Ninguna variante cumple la regla: no se pudo demostrar que agregar más de 2 meses de historia sume en esta ventana.')

print()
print(f'Rankings (sin corrección): {RANKINGS[0]} − {RANKINGS[1]} = {coma(rankings["d_media_201910_M"], 2)} M en 201910, '
      f'p {p_texto(rankings["p"])}, {rankings["semillas_a_favor"]} de 10 a favor.')
d_ref = (G.loc[(REFERENCIA, MES_PRINCIPAL)] - G.loc[(CONTROL, MES_PRINCIPAL)]).mean()
print(f'{REFERENCIA} − {CONTROL} (descriptiva): {coma(d_ref, 2)} M en 201910.')
print()
print('Lo que se analice a partir de acá es exploratorio y va en una sección aparte, rotulada así.')